# data cleaning

In [1]:
import pandas as pd

In [2]:
original_df = pd.read_csv('../dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')
original_df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
objects_list = original_df.select_dtypes(include=['object']).columns.to_list()
variation_df = pd.DataFrame(columns=['column_name', 'num_unique_values'])
for column in objects_list:
    unique_values = original_df[column].unique()
    num_unique_values = len(unique_values)
    new_row = pd.DataFrame({'column_name': column, 'num_unique_values': num_unique_values}, index=[0])
    variation_df =pd.concat([variation_df, new_row], ignore_index=True, axis=0, sort=False)
variation_df

,column_name,num_unique_values
0,customerID,7043
1,gender,2
2,Partner,2
3,Dependents,2
4,PhoneService,2
5,MultipleLines,3
6,InternetService,3
7,OnlineSecurity,3
8,OnlineBackup,3
9,DeviceProtection,3


In [4]:
df = original_df.replace({'TotalCharges': {' ':'0'}, 'SeniorCitizen': {0: 'No', 1: 'Yes'}})
total_charges_int_list = [float(x) for x in df['TotalCharges'].to_list()]
df['TotalCharges'] = total_charges_int_list
objects_list = [col for col in objects_list if col != 'TotalCharges' and col != 'customerID' and col != 'Churn']
objects_list.append('SeniorCitizen')
df.to_csv('../dataset/cleaned_churn_data.csv', index=False)

In [5]:
from sklearn.preprocessing import OneHotEncoder

df_dummies = pd.get_dummies(df[objects_list], drop_first=True)
encoder = OneHotEncoder(drop='first', sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[objects_list])
one_hot_df = pd.DataFrame(one_hot_encoded, 
                          columns=encoder.get_feature_names_out(objects_list))
df_encoded = pd.concat([df.drop(objects_list, axis=1), one_hot_df], axis=1)

In [6]:
df_encoded.head(5)

,customerID,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen_Yes
0,7590-VHVEG,1,29.85,29.85,No,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,5575-GNVDE,34,56.95,1889.50,No,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,3668-QPYBK,2,53.85,108.15,Yes,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,7795-CFOCW,45,42.30,1840.75,No,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,9237-HQITU,2,70.70,151.65,Yes,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [7]:
df_encoded.to_csv('../dataset/cleaned_encoded_churn_data.csv', index=False)

## baseline models

In [9]:
# logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = df_encoded.drop(['customerID', 'Churn'], axis=1)
y = df_encoded['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.preprocessing import StandardScaler

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.92      0.72      0.81      1036
         Yes       0.52      0.83      0.64       373

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409



In [11]:
from sklearn.metrics import roc_auc_score
y_proba = model.predict_proba(X_test)[:, 1]
print(f'Test AUC: {roc_auc_score(y_test, y_proba)}')

Test AUC: 0.8620946204726366
